# 🏗️ Entrenamiento LSTM — Agente de Gases Minería UPTC 2026

Este notebook genera datos sintéticos realistas, entrena modelos LSTM por zona
y exporta los archivos exactos que el backend espera:

- `lstm_gases_Frente_A_Sogamoso.keras`
- `lstm_gases_Frente_B_Mongua.keras`
- `lstm_gases_Galeria_Central.keras`
- `lstm_gases_Bocamina.keras`
- `lstm_scalers_gases_nuevos.pkl`

**Pasos:**
1. Ejecuta todas las celdas en orden (Runtime → Run all)
2. Al final descarga el ZIP con los 5 archivos
3. Copia los archivos a `modelos_reparados/gases/` en tu proyecto
4. Reinicia el Agente de Gases (`uvicorn backend.agentes.gases.app:app --port 8001`)

**Distribuciones usadas** — exactas a las del `simulador.py`:
```
Frente_A_Sogamoso: CH4(0.65,0.18) CO(17,5) CO2(0.21,0.07) O2(20.5,0.25) H2S(0.8,0.3)
Frente_B_Mongua:   CH4(0.50,0.15) CO(15,4) CO2(0.18,0.06) O2(20.6,0.20) H2S(0.6,0.25)
Galeria_Central:   CH4(0.45,0.12) CO(13,4) CO2(0.16,0.05) O2(20.7,0.18) H2S(0.5,0.20)
Bocamina:          CH4(0.30,0.10) CO(10,3) CO2(0.13,0.04) O2(20.8,0.15) H2S(0.3,0.15)
```

## 1. Instalar dependencias

In [ ]:
!pip install tensorflow scikit-learn numpy pandas matplotlib -q
import tensorflow as tf
import numpy as np
import pandas as pd
import pickle, os, zipfile, random
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

print(f'TensorFlow {tf.__version__}')
print('GPU disponible:', bool(tf.config.list_physical_devices("GPU")))

## 2. Configuración del modelo

In [ ]:
# ── Parámetros que DEBEN coincidir con backend/shared/config.py ──────────────
VENTANA    = 24    # lstm_ventana      — pasos de entrada
HORIZONTE  = 6     # lstm_horizonte    — pasos de salida
FEATURES   = ['CH4', 'CO', 'CO2', 'O2', 'H2S']
N_FEATURES = len(FEATURES)

# ── Perfiles por zona (igual que simulador.py → PERFILES_ZONA) ───────────────
PERFILES = {
    'Frente_A_Sogamoso': {'CH4':(0.22,0.07), 'CO':(5.5,1.8), 'CO2':(0.10,0.03), 'O2':(20.70,0.12), 'H2S':(0.18,0.07)},
    'Frente_B_Mongua':   {'CH4':(0.18,0.06), 'CO':(4.5,1.5), 'CO2':(0.09,0.03), 'O2':(20.75,0.10), 'H2S':(0.14,0.06)},
    'Galeria_Central':   {'CH4':(0.14,0.05), 'CO':(3.5,1.2), 'CO2':(0.08,0.02), 'O2':(20.80,0.09), 'H2S':(0.10,0.05)},
    'Bocamina':          {'CH4':(0.08,0.03), 'CO':(2.5,0.9), 'CO2':(0.06,0.02), 'O2':(20.85,0.08), 'H2S':(0.06,0.03)},
}

# ── Eventos críticos para enriquecer el entrenamiento ────────────────────────
EVENTOS_CRITICOS = [
    {'CH4': (1.5, 5.5)},                          # escape metano
    {'CO':  (80,  280), 'CO2': (1.2, 2.8)},       # incendio
    {'O2':  (16.5, 18.5)},                        # fuga agua / asphyxia
    {'CH4': (1.2, 3.0), 'CO': (60, 200)},         # explosión polvo
    {'H2S': (10, 50)},                            # fuga H2S
]

N_SERIE    = 5000    # pasos por zona (sin eventos)
N_EVENTOS  = 200     # pasos adicionales con eventos críticos
OUTPUT_DIR = '/content/modelos_entrenados'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Configuración OK')
print(f'  Ventana entrada : {VENTANA} pasos')
print(f'  Horizonte salida: {HORIZONTE} pasos')
print(f'  Features        : {FEATURES}')

## 3. Generación de datos sintéticos

In [ ]:
def generar_serie(zona, n_pasos=5000, n_eventos=200, seed=42):
    """
    Genera una serie temporal de gases para una zona.
    Incluye:
      - Tendencias suaves (seno de baja frecuencia)
      - Ruido gaussiano
      - Inyección de eventos críticos (5% de probabilidad)
      - Correlaciones realistas entre gases
    """
    rng = np.random.default_rng(seed)
    perfil = PERFILES[zona]
    t = np.arange(n_pasos + n_eventos)
    datos = []

    # Estado base con deriva temporal suave
    for i in t:
        fila = {}
        es_evento = (i >= n_pasos)  # últimos n_eventos son forzados

        # Valores normales con tendencia senoidal
        for gas, (mu, sigma) in perfil.items():
            tendencia = np.sin(2 * np.pi * i / 480) * sigma * 0.5  # ciclo ~2h
            val = rng.normal(mu + tendencia, sigma * 0.6)
            fila[gas] = float(np.clip(val, 0, None))

        # Correlación: si CH4 sube, CO2 también sube levemente
        if fila['CH4'] > perfil['CH4'][0] * 1.3:
            fila['CO2'] = fila['CO2'] * 1.15
            fila['O2']  = max(0, fila['O2'] - 0.05)

        # Eventos críticos al final de la serie (5% aleatorio en el resto)
        if es_evento or rng.random() < 0.05:
            ev = EVENTOS_CRITICOS[rng.integers(len(EVENTOS_CRITICOS))]
            for gas, (lo, hi) in ev.items():
                fila[gas] = float(rng.uniform(lo, hi))

        datos.append(fila)

    df = pd.DataFrame(datos, columns=FEATURES)
    return df


# Generar y visualizar
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, zona in zip(axes.flat, PERFILES.keys()):
    df = generar_serie(zona)
    ax.plot(df['CH4'].values[:500], label='CH4 (%)', alpha=0.8)
    ax.plot(df['CO'].values[:500] / 100, label='CO/100 (ppm)', alpha=0.7)
    ax.set_title(zona.replace('_', ' '))
    ax.legend(fontsize=8)
    ax.set_xlabel('Pasos')

plt.suptitle('Series temporales de gases por zona (primeros 500 pasos)', fontsize=13)
plt.tight_layout()
plt.show()
print('Datos generados OK')

## 4. Preparar ventanas de entrenamiento

In [ ]:
def crear_ventanas(serie_norm, ventana, horizonte):
    """
    Crea pares (X, y):
      X: (n_muestras, ventana, n_features)
      y: (n_muestras, horizonte, n_features)
    """
    X, y = [], []
    n = len(serie_norm)
    for i in range(n - ventana - horizonte + 1):
        X.append(serie_norm[i : i + ventana])
        y.append(serie_norm[i + ventana : i + ventana + horizonte])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


# Vista previa de shapes
zona_test = 'Frente_A_Sogamoso'
df_test = generar_serie(zona_test)
sc_test = StandardScaler()
datos_norm = sc_test.fit_transform(df_test.values)
X_t, y_t = crear_ventanas(datos_norm, VENTANA, HORIZONTE)
print(f'Forma X: {X_t.shape}  →  (muestras, ventana={VENTANA}, features={N_FEATURES})')
print(f'Forma y: {y_t.shape}  →  (muestras, horizonte={HORIZONTE}, features={N_FEATURES})')
print(f'Train/Val split 80/20: {int(len(X_t)*0.8)} / {int(len(X_t)*0.2)} muestras')

## 5. Arquitectura del modelo LSTM

In [ ]:
def construir_modelo(ventana, n_features, horizonte):
    """
    LSTM apilado con salida de secuencia completa (horizonte x n_features).
    Arquitectura balanceada: buena precisión sin overfitting.
    """
    modelo = Sequential([
        Input(shape=(ventana, n_features)),
        LSTM(128, return_sequences=True),
        Dropout(0.2),
        LSTM(64, return_sequences=False),
        Dropout(0.2),
        Dense(horizonte * n_features, activation='linear'),
    ])
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='mse',
        metrics=['mae']
    )
    return modelo

modelo_preview = construir_modelo(VENTANA, N_FEATURES, HORIZONTE)
modelo_preview.summary()
del modelo_preview

## 6. Entrenamiento por zona (la celda principal)

In [ ]:
scalers_dict = {}   # {zona: StandardScaler}
historiales  = {}   # para graficar después

for zona in PERFILES.keys():
    print(f'\n{'='*55}')
    print(f'  Entrenando: {zona}')
    print('='*55)

    # 1) Generar datos (seed diferente por zona para variedad)
    seed = {'Frente_A_Sogamoso':1,'Frente_B_Mongua':2,'Galeria_Central':3,'Bocamina':4}[zona]
    df = generar_serie(zona, n_pasos=5000, n_eventos=200, seed=seed)

    # 2) Normalizar (StandardScaler sobre el set completo)
    scaler = StandardScaler()
    datos_norm = scaler.fit_transform(df.values).astype(np.float32)
    scalers_dict[zona] = scaler

    # 3) Crear ventanas
    X, y = crear_ventanas(datos_norm, VENTANA, HORIZONTE)
    # y debe ser (n, horizonte*n_features) para la capa Dense
    y_flat = y.reshape(len(y), HORIZONTE * N_FEATURES)

    # 4) Train / Val split
    split = int(len(X) * 0.8)
    X_tr, X_val = X[:split], X[split:]
    y_tr, y_val = y_flat[:split], y_flat[split:]

    print(f'  Train: {len(X_tr)} muestras | Val: {len(X_val)} muestras')

    # 5) Construir y entrenar
    modelo = construir_modelo(VENTANA, N_FEATURES, HORIZONTE)

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5, verbose=1),
    ]

    hist = modelo.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=60,
        batch_size=64,
        callbacks=callbacks,
        verbose=1,
    )
    historiales[zona] = hist

    # 6) Evaluar en validación
    pred_norm = modelo.predict(X_val, verbose=0)
    pred_norm_3d = pred_norm.reshape(-1, HORIZONTE, N_FEATURES)
    y_val_3d    = y_val.reshape(-1, HORIZONTE, N_FEATURES)

    # Desnormalizar para MAE en unidades reales
    pred_real = scaler.inverse_transform(pred_norm_3d.reshape(-1, N_FEATURES))
    y_real    = scaler.inverse_transform(y_val_3d.reshape(-1, N_FEATURES))
    mae_global = mean_absolute_error(y_real, pred_real)

    print(f'\n  MAE global (val, unidades reales): {mae_global:.4f}')
    for i, gas in enumerate(FEATURES):
        mae_gas = mean_absolute_error(y_real[:, i], pred_real[:, i])
        print(f'    {gas}: MAE = {mae_gas:.4f}')

    # 7) Guardar modelo
    ruta = f'{OUTPUT_DIR}/lstm_gases_{zona}.keras'
    modelo.save(ruta)
    print(f'  ✓ Guardado: {ruta}')

print('\n✅ Todos los modelos entrenados.')

## 7. Guardar scalers

In [ ]:
ruta_scalers = f'{OUTPUT_DIR}/lstm_scalers_gases_nuevos.pkl'
with open(ruta_scalers, 'wb') as f:
    pickle.dump(scalers_dict, f)

print(f'Scalers guardados: {ruta_scalers}')
print('Zonas incluidas:', list(scalers_dict.keys()))

# Verificar round-trip
with open(ruta_scalers, 'rb') as f:
    sc_check = pickle.load(f)

print('\nVerificación round-trip de scalers:')
for zona, sc in sc_check.items():
    X_dummy = np.array([[0.5, 15, 0.2, 20.8, 0.5]], dtype=np.float32)
    X_norm  = sc.transform(X_dummy)
    X_back  = sc.inverse_transform(X_norm)
    err     = np.max(np.abs(X_dummy - X_back))
    print(f'  {zona}: error max = {err:.2e}  "OK" if err < 1e-5 else "ERROR"')

## 8. Validar predicciones end-to-end

In [ ]:
print('Prueba de predicción end-to-end (simula lo que hace predictor.py):')
print()

# tf already imported

for zona in PERFILES.keys():
    # Cargar modelo desde disco
    modelo_cargado = tf.keras.models.load_model(
        f'{OUTPUT_DIR}/lstm_gases_{zona}.keras', compile=False)

    # Simular historial de 30 lecturas normales
    perfil = PERFILES[zona]
    historial = [
        {g: float(np.clip(np.random.normal(mu, sigma), 0, None))
         for g, (mu, sigma) in perfil.items()}
        for _ in range(30)
    ]

    # Normalizar ventana de 24
    sc = sc_check[zona]
    X_raw = np.array([[h[g] for g in FEATURES] for h in historial[-VENTANA:]], dtype=np.float32)
    X_norm = sc.transform(X_raw).reshape(1, VENTANA, N_FEATURES)

    # Predecir
    pred_norm = modelo_cargado.predict(X_norm, verbose=0)[0]  # (HORIZONTE*N_FEATURES,)
    pred_norm_3d = pred_norm.reshape(HORIZONTE, N_FEATURES)

    # Desnormalizar
    pred_real = sc.inverse_transform(pred_norm_3d)
    pred_real = np.clip(pred_real, 0, None)

    print(f'{zona}:')
    for t in range(HORIZONTE):
        vals = dict(zip(FEATURES, pred_real[t].round(3)))
        print(f'  +{(t+1)*15:3d} min → {vals}')
    print()

## 9. Curvas de entrenamiento

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, (zona, hist) in zip(axes.flat, historiales.items()):
    ax.plot(hist.history['loss'],     label='Train Loss')
    ax.plot(hist.history['val_loss'], label='Val Loss')
    ax.set_title(zona.replace('_', ' '))
    ax.set_xlabel('Época')
    ax.set_ylabel('MSE')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Curvas de entrenamiento LSTM por zona', fontsize=13)
plt.tight_layout()
plt.show()

## 10. Empaquetar y descargar

In [ ]:
zip_path = '/content/modelos_gases_lstm.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for archivo in os.listdir(OUTPUT_DIR):
        ruta_completa = os.path.join(OUTPUT_DIR, archivo)
        zf.write(ruta_completa, arcname=archivo)
        print(f'  + {archivo}')

print(f'\nZIP creado: {zip_path}')
print(f'Tamaño: {os.path.getsize(zip_path) / 1024 / 1024:.1f} MB')

# Descarga automática en Colab
try:
    from google.colab import files
    files.download(zip_path)
    print('\n✅ Descarga iniciada. Guarda el ZIP y extrae los archivos en:')
    print('   proyecto_mineria_ia/modelos_reparados/gases/')
except ImportError:
    print(f'\nArchivos disponibles en: {OUTPUT_DIR}')

## ✅ Pasos para instalar los modelos en el proyecto

```
1. Extrae el ZIP descargado

2. Copia los 5 archivos a:
   proyecto_mineria_ia/
   └── modelos_reparados/
       └── gases/
           ├── lstm_gases_Frente_A_Sogamoso.keras   ← reemplaza
           ├── lstm_gases_Frente_B_Mongua.keras     ← reemplaza
           ├── lstm_gases_Galeria_Central.keras     ← reemplaza
           ├── lstm_gases_Bocamina.keras            ← reemplaza
           └── lstm_scalers_gases_nuevos.pkl        ← reemplaza

3. Reinicia el Agente de Gases:
   uvicorn backend.agentes.gases.app:app --host 0.0.0.0 --port 8001 --reload

4. En los logs debes ver:
   Scalers LSTM cargados para: ['Bocamina', 'Frente_A_Sogamoso', ...]
   Cargado: lstm_gases_Frente_A_Sogamoso.keras
   ...
   Predictor LSTM listo: 4/4 zonas

5. Inicia la simulación y espera ~6 min (24 ciclos × 15 s)
   para que el panel de predicciones muestre resultados.
```

### ¿Por qué los modelos anteriores no funcionaban?
Los scalers guardados tenían media CH4=1.08%, O2=24.77% — valores que
no corresponden a la distribución del simulador (CH4~0.5%, O2~20.6%).
El modelo normalizaba las entradas con la distribución incorrecta,
produciendo predicciones fuera de rango que el sistema descartaba.